In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
import scipy.io
import os
import glob

# =============================================================================
# 1. 特征提取核心函数
# =============================================================================
def calculate_time_features(signal: pd.Series) -> dict:
    """计算一维信号的时域特征"""
    signal = signal.dropna()
    rms = np.sqrt(np.mean(signal**2))
    sqrt_amp = (np.mean(np.sqrt(np.abs(signal))))**2
    
    features = {
        'Mean': np.mean(signal), 'RMS': rms, 'Var': np.var(signal),
        'Skew': skew(signal), 'Kurt': kurtosis(signal),
        'CF': np.max(np.abs(signal)) / rms if rms != 0 else 0,
        'MF': np.max(np.abs(signal)) / sqrt_amp if sqrt_amp != 0 else 0,
        'P2P': np.max(signal) - np.min(signal),
    }
    return features

def calculate_freq_features(signal: pd.Series, sampling_rate: int) -> dict:
    """计算一维信号的频域特征"""
    signal = signal.dropna()
    n_points = len(signal)
    if n_points == 0:
        return {'FreqMean': 0, 'FreqSTD': 0, 'FreqSkew': 0, 'FreqKurt': 0}

    fft_vals = np.fft.fft(signal)
    fft_freq = np.fft.fftfreq(n_points, 1.0 / sampling_rate)
    positive_freq_indices = np.where(fft_freq >= 0)
    freqs = fft_freq[positive_freq_indices]
    amplitudes = np.abs(fft_vals[positive_freq_indices])
    power_spectrum = amplitudes**2
    power_spectrum_normalized = power_spectrum / np.sum(power_spectrum) if np.sum(power_spectrum) != 0 else power_spectrum
    
    freq_mean = np.sum(freqs * power_spectrum_normalized)
    freq_std = np.sqrt(np.sum(((freqs - freq_mean)**2) * power_spectrum_normalized))
    epsilon = 1e-10
    freq_skew = np.sum(((freqs - freq_mean) / (freq_std + epsilon))**3 * power_spectrum_normalized)
    freq_kurt = np.sum(((freqs - freq_mean) / (freq_std + epsilon))**4 * power_spectrum_normalized) - 3

    features = {
        'FreqMean': freq_mean, 'FreqSTD': freq_std,
        'FreqSkew': freq_skew, 'FreqKurt': freq_kurt
    }
    return features

def process_mat_file_to_features(mat_file_path: str, sampling_rate: int = 12000) -> pd.DataFrame:
    """
    加载单个mat文件，提取信号和RPM，并计算所有特征。
    """
    try:
        data = scipy.io.loadmat(mat_file_path)
    except Exception as e:
        print(f"  -> 读取文件 {mat_file_path} 失败: {e}")
        return pd.DataFrame()
        
    # --- 新增代码：提取RPM值 ---
    rpm_value = None
    # 查找以'RPM'结尾的键
    rpm_keys = [key for key in data.keys() if key.endswith('RPM')]
    if rpm_keys:
        # 假设只有一个RPM键，并提取其值
        # .mat文件中的单个数值通常被包裹在多维数组中，例如 [[1797]]
        rpm_value = data[rpm_keys[0]][0][0]
    # --- RPM提取结束 ---

    signal_keys = [key for key in data.keys() if key.endswith('_time')]
    if not signal_keys:
        return pd.DataFrame()
        
    signal_df = pd.DataFrame({key: data[key].flatten() for key in signal_keys})

    results = []
    for col_name in signal_df.columns:
        signal = signal_df[col_name]
        time_feats = calculate_time_features(signal)
        freq_feats = calculate_freq_features(signal, sampling_rate)
        all_feats = {**time_feats, **freq_feats}
        all_feats['Name'] = col_name
        
        # --- 新增代码：将RPM值添加到每一行特征中 ---
        all_feats['RPM'] = rpm_value
        # --- 添加结束 ---

        results.append(all_feats)
        
    return pd.DataFrame(results)

# =============================================================================
# 2. 主程序：遍历、处理、整合
# =============================================================================
def process_all_data(root_dir: str, sampling_rate: int = 12000):
    """
    遍历指定根目录下的所有子文件夹，找到.mat文件，提取特征，并记录文件层级。
    """
    if not os.path.isdir(root_dir):
        print(f"错误：提供的路径 '{root_dir}' 不是一个有效的文件夹。")
        return

    all_rows_list = []
    
    print("开始扫描文件...")
    # Step 1: 遍历所有.mat文件，提取特征并记录原始路径
    for dirpath, _, filenames in os.walk(root_dir):
        # 对文件名进行排序，确保处理顺序一致
        for filename in sorted(filenames):
            if filename.endswith('.mat'):
                full_path = os.path.join(dirpath, filename)
                
                relative_path = os.path.relpath(full_path, root_dir)
                path_parts = relative_path.split(os.sep)
                
                features_df = process_mat_file_to_features(full_path, sampling_rate)
                
                if not features_df.empty:
                    features_df['PathParts'] = [path_parts] * len(features_df)
                    all_rows_list.append(features_df)

    if not all_rows_list:
        print("扫描完成，但在指定文件夹下未找到或未能成功处理任何 .mat 文件。")
        return
        
    # Step 2: 合并所有数据
    final_df = pd.concat(all_rows_list, ignore_index=True)

    # Step 3: 根据路径列表创建层级列
    max_depth = final_df['PathParts'].apply(len).max()
    level_cols = [f'Level_{i+1}' for i in range(max_depth)]
    path_df = pd.DataFrame(final_df['PathParts'].tolist(), index=final_df.index, columns=level_cols)
    final_df = pd.concat([path_df, final_df], axis=1)
    final_df = final_df.drop('PathParts', axis=1)
    
    # Step 4: 处理特殊的"OR"文件夹层级
    print("正在处理'OR'文件夹的特殊层级结构...")
    or_sublevel_col = 'OR_SubLevel'
    final_df[or_sublevel_col] = pd.NA

    fault_type_col = None
    for col in level_cols:
        if 'OR' in final_df[col].astype(str).unique():
            fault_type_col = col
            break
            
    if fault_type_col:
        or_level_index = int(fault_type_col.split('_')[1])
        subsequent_cols = [f'Level_{i+1}' for i in range(or_level_index, max_depth - 1)]

        for i, row in final_df.iterrows():
            if row[fault_type_col] == 'OR' and or_level_index < max_depth:
                or_sub_folder_col = f'Level_{or_level_index+1}'
                if or_sub_folder_col in final_df.columns:
                    final_df.at[i, or_sublevel_col] = row[or_sub_folder_col]
                    
                    for j in range(len(subsequent_cols)):
                        current_col = subsequent_cols[j]
                        next_col_index = or_level_index + j + 2
                        if f'Level_{next_col_index}' in final_df.columns:
                            final_df.at[i, current_col] = row[f'Level_{next_col_index}']
                        else:
                            final_df.at[i, current_col] = pd.NA
    
    # Step 5: 排序
    print("正在对数据进行排序...")
    sort_columns = [col for col in final_df.columns if col.startswith('Level_') or col == or_sublevel_col]
    sort_columns.append('Name')
    final_df = final_df.sort_values(by=sort_columns).reset_index(drop=True)

    # Step 6: 整理最终列顺序并保存
    # --- 更新代码：将RPM添加到特征列列表中 ---
    feature_cols = ['RPM', 'Mean', 'RMS', 'Var', 'Skew', 'Kurt', 'CF', 'MF', 'P2P', 
                    'FreqMean', 'FreqSTD', 'FreqSkew', 'FreqKurt']
    
    final_cols = sort_columns + feature_cols
    final_cols = [col for col in final_cols if col in final_df.columns]
    final_df = final_df[final_cols]

    output_filename = '../data/features/source/all_data_features_sorted_with_rpm.csv'
    final_df.to_csv(output_filename, index=False)
    
    print("\n处理完成！")
    print(f"所有特征已汇总、排序并保存到文件: {output_filename}")
    print("\n最终输出数据预览:")
    print(final_df.head().to_string())


# --- 如何使用 ---
if __name__ == "__main__":
    # ** 请在这里修改为您存放赛方数据的根文件夹路径 **
    root_directory = '../data/raw/source' # '.' 代表当前文件夹

    process_all_data(root_directory)

开始扫描文件...
正在处理'OR'文件夹的特殊层级结构...
正在对数据进行排序...

处理完成！
所有特征已汇总、排序并保存到文件: all_data_features_sorted_with_rpm.csv

最终输出数据预览:
         Level_1 Level_2 Level_3     Level_4 Level_5 OR_SubLevel          Name   RPM      Mean       RMS       Var      Skew      Kurt        CF        MF       P2P     FreqMean      FreqSTD  FreqSkew  FreqKurt
0  12kHz_DE_data       B    0007  B007_0.mat    None        <NA>  X118_BA_time  1796 -0.000006  0.035296  0.001246  0.030537  0.124039  4.531553  6.745104  0.315622  1731.238417  1414.860339  0.885545 -0.763447
1  12kHz_DE_data       B    0007  B007_0.mat    None        <NA>  X118_DE_time  1796  0.000144  0.137790  0.018986 -0.006781 -0.041165  4.496886  6.642001  1.199096  2998.682419   741.809794 -2.363447  5.118996
2  12kHz_DE_data       B    0007  B007_0.mat    None        <NA>  X118_FE_time  1796 -0.000010  0.105304  0.011089  0.007305 -0.237246  3.979010  5.764744  0.823873  3294.230017  1463.053586 -0.954363 -0.623663
3  12kHz_DE_data       B    0007  B00